In [29]:
import json
import os
from pathlib import Path
from typing import List, Optional

import boto3
import instructor
from botocore.config import Config as BotoConfig
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel, Field
from qdrant_client import QdrantClient

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
load_dotenv(repo_root / ".env")
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_DEFAULT_REGION = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "cohere.embed-v4")
QDRANT_URL = os.getenv("QDRANT_URL", "http://213.136.80.53:6333")
QDRANT_COLLECTION = os.getenv("QDRANT_COLLECTION", "legal_acts_event_rag_full")
GEMINI_MODEL = os.getenv("DEFAULT_MODEL_NAME", "gemini-2.5-flash")

required = {
    "GEMINI_API_KEY": GEMINI_API_KEY,
    "AWS_ACCESS_KEY_ID": AWS_ACCESS_KEY_ID,
    "AWS_SECRET_ACCESS_KEY": AWS_SECRET_ACCESS_KEY,
}
missing = [name for name, value in required.items() if not value]
if missing:
    raise ValueError(f"Missing required env vars: {missing}")

print(
    {
        "gemini_model": GEMINI_MODEL,
        "embedding_model": EMBEDDING_MODEL,
        "qdrant_url": QDRANT_URL,
        "qdrant_collection": QDRANT_COLLECTION,
    }
)

{'gemini_model': 'gemini-2.5-flash', 'embedding_model': 'cohere.embed-multilingual-v3', 'qdrant_url': 'http://213.136.80.53:6333', 'qdrant_collection': 'legal_acts_event_rag_full'}


In [21]:
def get_bedrock_client():
    return boto3.client(
        service_name="bedrock-runtime",
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        region_name=AWS_DEFAULT_REGION,
        config=BotoConfig(retries={"mode": "adaptive"}),
    )


def embed_text_query(text: str, max_input_chars: int = 2048) -> List[float]:
    query_text = text[:max_input_chars]
    body = {
        "input_type": "search_query",
        "embedding_types": ["float"],
        "texts": [query_text],
    }
    response = get_bedrock_client().invoke_model(
        modelId=EMBEDDING_MODEL,
        body=json.dumps(body),
        accept="application/json",
        contentType="application/json",
    )
    payload = json.loads(response["body"].read())

    embeddings = payload.get("embeddings")
    if isinstance(embeddings, dict) and "float" in embeddings:
        return embeddings["float"][0]
    if isinstance(embeddings, list):
        if embeddings and isinstance(embeddings[0], list):
            return embeddings[0]
        return embeddings
    if "embedding" in payload:
        return payload["embedding"]
    raise ValueError(f"Embedding missing in Bedrock response: {payload}")

In [22]:
qdrant = QdrantClient(url=QDRANT_URL, timeout=30)
collection_info = qdrant.get_collection(QDRANT_COLLECTION)

vectors_count = getattr(collection_info, "vectors_count", None)
if (
    vectors_count is None
    and hasattr(collection_info, "config")
    and collection_info.config is not None
):
    vectors_cfg = getattr(collection_info.config.params, "vectors", None)
    if isinstance(vectors_cfg, dict):
        vectors_count = len(vectors_cfg)
    elif vectors_cfg is not None:
        vectors_count = 1

print(
    {
        "status": str(collection_info.status),
        "points_count": getattr(collection_info, "points_count", None),
        "vectors_count": vectors_count,
    }
)

{'status': 'green', 'points_count': 44859, 'vectors_count': 1}


In [23]:
def retrieve_sources(question: str, top_k: int = 6) -> List[dict]:
    vector = embed_text_query(question, max_input_chars=2048)
    hits = qdrant.query_points(
        collection_name=QDRANT_COLLECTION,
        query=vector,
        limit=top_k,
        with_payload=True,
    ).points

    sources: List[dict] = []
    for idx, hit in enumerate(hits, start=1):
        payload = hit.payload or {}
        excerpt = str(payload.get("section_content_clean") or "").strip()
        if not excerpt:
            excerpt = "No excerpt available."
        sources.append(
            {
                "citation_id": idx,
                "act_title": payload.get("act_title"),
                "act_year": payload.get("act_year"),
                "section_index": (
                    str(payload.get("section_index"))
                    if payload.get("section_index") is not None
                    else None
                ),
                "source_url": payload.get("source_url"),
                "excerpt": excerpt,
                "score": float(hit.score or 0.0),
            }
        )
    return sources


def build_grounded_prompt(question: str, sources: List[dict]) -> List[dict]:
    context_blocks = []
    for source in sources:
        context_blocks.append(
            "\n".join(
                [
                    f"[Source {source['citation_id']}]",
                    f"Act: {source['act_title'] or 'Unknown'}",
                    f"Year: {source['act_year'] if source['act_year'] is not None else 'Unknown'}",
                    f"Section: {source['section_index'] or 'Unknown'}",
                    f"Text: {source['excerpt']}",
                    f"URL: {source['source_url'] or 'N/A'}",
                ]
            )
        )

    system_prompt = (
        "You are a legal research assistant for Bangladesh law. "
        "Answer only from the supplied legal sources. "
        "If sources are insufficient, explicitly say what is missing. "
        "Every substantive claim must include source citations like [Source 1]. "
        "Do not fabricate statutes, sections, or outcomes."
    )
    user_prompt = (
        f"Question: {question}\n\n"
        f"Legal sources:\n{chr(10).join(context_blocks)}\n\n"
        "Provide:\n"
        "1) Direct answer in plain language.\n"
        "2) Brief legal basis with citation tags.\n"
        "3) If uncertain, mention limits clearly."
    )

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

In [ ]:
class GroundedClaim(BaseModel):
    claim: str = Field(description="A concrete legal statement")
    source_ids: List[int] = Field(
        description="One or more source ids supporting this claim"
    )


class StructuredLegalAnswer(BaseModel):
    direct_answer: str = Field(description="Short direct answer to the user question")
    legal_basis: List[GroundedClaim] = Field(
        description="Grounded legal reasoning points"
    )
    limitations: Optional[str] = Field(
        default=None, description="Missing context or uncertainty"
    )

In [30]:
genai_client = genai.Client(api_key=GEMINI_API_KEY)
structured_client = instructor.from_genai(
    genai_client,
    model=GEMINI_MODEL,
    mode=instructor.Mode.GENAI_STRUCTURED_OUTPUTS,
)
print({"model": GEMINI_MODEL})

{'model': 'gemini-2.5-flash'}


In [32]:
question = "What is the punishment for theft under Bangladesh law?"
sources = retrieve_sources(question, top_k=6)

if not sources:
    raise ValueError(
        "I could not find relevant legal sources in the vector store for this question."
    )

messages = build_grounded_prompt(question, sources)

structured_answer = structured_client.create(
    response_model=StructuredLegalAnswer,
    messages=[
        messages[0],
        {
            "role": "user",
            "content": messages[1]["content"]
            + "\n\nReturn strict JSON matching the response schema.",
        },
    ],
)

structured_answer.model_dump()

{'direct_answer': 'The provided legal sources do not contain information regarding the punishment for theft under Bangladesh law. All the supplied sections of The Penal Code, 1860, pertain to offenses related to counterfeiting coins and related activities.',
 'legal_basis': [],
 'limitations': 'The provided legal sources are insufficient to determine the punishment for theft under Bangladesh law. The sources exclusively cover offenses related to counterfeiting currency, not theft.'}

In [27]:
for source in sources:
    print(f"Source {source['citation_id']} | score={source['score']:.4f}")
    print(f"Act: {source.get('act_title')} | Section: {source.get('section_index')}")
    print((source.get("excerpt") or "")[:350])
    print("-" * 80)

Source 1 | score=0.6979
Act: 1The Penal Code, 1860 | Section: 238
238. Whoever imports into Bangladesh, or exports therefrom, any counterfeit coin which he knows or has reason to believe to be a counterfeit of Bangladesh coin, shall be punished withimprisonment for life, or with imprisonment of either description for a term which may extend to ten years, and shall also be liable to fine.
--------------------------------------------------------------------------------
Source 2 | score=0.6977
Act: 1The Penal Code, 1860 | Section: 232
232. Whoever counterfeits, or knowingly performs any part of the process of counterfeiting Bangladesh coin, shall be punished with imprisonment for life, or withimprisonment of either description for a term which may extend to ten years, and shall also be liable to fine.
--------------------------------------------------------------------------------
Source 3 | score=0.6831
Act: 1The Penal Code, 1860 | Section: 245
245. Whoever, without lawful authority, tak

In [33]:
hard_question = (
    "What are Bangladesh regulations for interplanetary mining permits on Mars?"
)
hard_sources = retrieve_sources(hard_question, top_k=4)

hard_messages = build_grounded_prompt(hard_question, hard_sources)

hard_answer = structured_client.create(
    response_model=StructuredLegalAnswer,
    messages=[
        hard_messages[0],
        {
            "role": "user",
            "content": hard_messages[1]["content"]
            + "\n\nReturn strict JSON matching the response schema.",
        },
    ],
)

hard_answer.model_dump()

{'direct_answer': 'The provided legal sources do not contain any regulations regarding interplanetary mining permits on Mars for Bangladesh.',
 'legal_basis': [],
 'limitations': "The provided legal sources exclusively pertain to maritime law, covering aspects such as activities in the international seabed, coastal trade, ship equipment, and provisions on vessels within or related to Bangladesh's national jurisdiction [Source 1, Source 2, Source 3, Source 4]. They do not address space law, interplanetary mining, or activities on celestial bodies like Mars. Therefore, there is insufficient information to answer the question about Bangladesh regulations for interplanetary mining permits on Mars."}